<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/how-to-train-your-llm-notebooks/blob/main/12-the-bill.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/how-to-train-your-llm/) collection._

# Chapter 12 — What Training One Costs, and When You Should Not

Companion to [the chapter](https://www.ai.biz/books/how-to-train-your-llm/what-training-one-costs-and-when-you-should-not/).

No model is trained here. The notebook is a cost sheet: every assumption is a named variable at the top of a cell, so that the arithmetic can be rerun with your own numbers. It runs in under a second.

## 1. The run itself

The 6ND estimate from chapter 1, with the hardware and price as assumptions and two multipliers for what the estimate leaves out.

In [ ]:
import math

GPU_PEAK_TFLOPS = 989          # H100 SXM, dense bf16
MFU = 0.40                     # model FLOPs utilisation: the share of peak a good training run sustains
PRICE_PER_GPU_HOUR = 2.0       # dollars, roughly the 2026 rental rate for an H100
CLUSTER = 1024                 # GPUs available

def run_cost(N, D, mfu=MFU, overhead=1.0):
    """GPU-hours, days on the cluster and dollars for a run of N parameters on D tokens."""
    flops = 6 * N * D * overhead
    gpu_hours = flops / (GPU_PEAK_TFLOPS * 1e12 * mfu) / 3600
    return gpu_hours, gpu_hours / CLUSTER / 24, gpu_hours * PRICE_PER_GPU_HOUR

print(f'{"run":<26} {"params":>7} {"tokens":>7} {"GPU-hours":>10} {"days on 1,024":>13} {"cost":>12}')
runs = [('1B, Chinchilla-optimal', 1e9, 20e9), ('8B, Chinchilla-optimal', 8e9, 160e9), ('8B, Llama 3 recipe', 8e9, 15e12),
        ('70B, Llama 3 recipe', 70e9, 15e12), ('405B, Llama 3 recipe', 405e9, 15.6e12)]
for name, N, D in runs:
    h, days, cost = run_cost(N, D)
    tokens = f'{D / 1e12:.1f}T' if D >= 1e12 else f'{D / 1e9:.0f}B'
    print(f'{name:<26} {N / 1e9:>6.0f}B {tokens:>7} {h:>10,.0f} {days:>13.1f} ${cost:>11,.0f}')

print()
print(f'{"the 70B run, with":<40} {"cost":>12}')
for label, mfu, overhead in [('the arithmetic as above', 0.40, 1.0), ('a worse utilisation, 30%', 0.30, 1.0),
                             ('restarts and lost work, +15%', 0.30, 1.15), ('the small-scale experiments, +40%', 0.30, 1.6),
                             ('one failed full-scale attempt', 0.30, 2.6)]:
    print(f'{label:<40} ${run_cost(70e9, 15e12, mfu, overhead)[2]:>11,.0f}')

## 2. The rest of the bill

Everything that is spent before the run starts and after it ends. The staff figures are loaded costs per person-month and are assumptions; change them.

In [ ]:
PERSON_MONTH = 25_000           # loaded cost of one engineer or researcher for one month
STORAGE_PER_TB_MONTH = 20       # object storage
EVAL_GPU_HOURS = 5_000          # benchmarks, ablations and sanity checks across the run

def full_bill(N, D, data_months, team, team_months, raw_tb, checkpoints_tb, months_kept=12):
    h, days, run = run_cost(N, D, mfu=0.30, overhead=1.6)
    lines = {
        'training run, with overheads': run,
        'data pipeline (people)': data_months * PERSON_MONTH,
        'data storage, raw and processed': raw_tb * STORAGE_PER_TB_MONTH * months_kept,
        'checkpoint storage': checkpoints_tb * STORAGE_PER_TB_MONTH * months_kept,
        'evaluation compute': EVAL_GPU_HOURS * PRICE_PER_GPU_HOUR,
        'training and infrastructure team': team * team_months * PERSON_MONTH,
    }
    return lines

for name, N, D, kw in [('8B on 15T tokens', 8e9, 15e12, dict(data_months=18, team=6, team_months=6, raw_tb=200, checkpoints_tb=20)),
                       ('70B on 15T tokens', 70e9, 15e12, dict(data_months=36, team=15, team_months=9, raw_tb=200, checkpoints_tb=150))]:
    lines = full_bill(N, D, **kw)
    total = sum(lines.values())
    print(f'{name}')
    for k, v in lines.items():
        print(f'  {k:<36} ${v:>12,.0f} {v / total:>6.1%}')
    print(f'  {"total":<36} ${total:>12,.0f}\n')

## 3. Buying the tokens instead

The alternative to training is renting someone else's model by the token. The break-even depends on volume, and the serving arithmetic is chapter 11's.

In [ ]:
API_PRICE_PER_M_OUTPUT = 2.00   # assumption: dollars per million output tokens from a hosted model of comparable size
API_PRICE_PER_M_INPUT = 0.50
OWN_COST_PER_M = 0.25           # chapter 11's figure for a 70B model served at batch 32 on rented H100s, plus a margin for idle time

monthly_tokens = [1e8, 1e9, 1e10, 1e11]
print(f'{"tokens a month":>14} {"buy, per month":>14} {"serve your own":>14} {"saving a month":>14}')
for t in monthly_tokens:
    buy = t / 1e6 * (API_PRICE_PER_M_OUTPUT * 0.5 + API_PRICE_PER_M_INPUT * 0.5)
    own = t / 1e6 * OWN_COST_PER_M
    print(f'{t:>14.0e} ${buy:>13,.0f} ${own:>13,.0f} ${buy - own:>13,.0f}')

train_70b = sum(full_bill(70e9, 15e12, data_months=36, team=15, team_months=9, raw_tb=200, checkpoints_tb=150).values())
saving_per_token = (API_PRICE_PER_M_OUTPUT * 0.5 + API_PRICE_PER_M_INPUT * 0.5 - OWN_COST_PER_M) / 1e6
print(f'\nthe 70B bill of ${train_70b:,.0f} is repaid after {train_70b / saving_per_token:.2e} tokens,')
months = train_70b / saving_per_token / 1e10
print(f'which at 10 billion tokens a month is {months:,.0f} months, or {months / 12:,.0f} years, before the model is out of date')

## 4. Adapting one instead

The third option, which the next book is about: take a trained base model and fine-tune it. The same arithmetic, with the token counts fine-tuning actually uses. LoRA skips the gradients of the frozen weights, about a third of the work.

In [ ]:
print(f'{"job":<34} {"FLOPs":>9} {"GPU-hours":>10} {"cost":>10}')
for name, N, D, factor in [('pre-train 8B on 15T tokens', 8e9, 15e12, 6), ('full fine-tune 8B on 1B tokens', 8e9, 1e9, 6),
                           ('LoRA fine-tune 8B on 1B tokens', 8e9, 1e9, 4), ('full fine-tune 70B on 1B tokens', 70e9, 1e9, 6)]:
    flops = factor * N * D
    h = flops / (GPU_PEAK_TFLOPS * 1e12 * 0.40) / 3600
    print(f'{name:<34} {flops:>9.1e} {h:>10,.0f} ${h * PRICE_PER_GPU_HOUR:>9,.0f}')

## Try it yourself

1. Put your own numbers into section 1: the GPU you can actually rent, its price, and the utilisation your team has achieved before. How far does the 70B figure move?
2. Section 3 assumes the hosted price stays where it is. Halve it every twelve months and recompute the months to break even.
3. Add a line to section 2 for the people who use the model after it exists: evaluation, safety review, serving operations. Which section of the bill is largest now?